# 05 — Innovation Classification

## Purpose

Classify retrieved World Bank project components and subcomponents using a structured innovation ontology.

Notebook:

1. Loads candidate analytical units identified through semantic retrieval.
2. Links candidates back to their full component or subcomponent text.
3. Applies an ontology-based LLM classifier.
4. Produces structured labels for innovation presence, type, novelty, maturity, and evidence strength.
5. Saves classifications for downstream expert validation and error analysis.

Semantic retrieval identifies candidate innovation signals. It does not determine whether an activity is innovative.

In [ ]:
from pathlib import Path
import json
import os
import time
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, Field


# --------------------------------------------------
# PROJECT CONFIGURATION
# --------------------------------------------------

ROOT = Path("..").resolve()

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
ONTOLOGY_DIR = ROOT / "ontology"
OUTPUT_DIR = ROOT / "outputs"

CANDIDATE_PATH = PROCESSED_DIR / "innovation_candidates.csv"
UNITS_PATH = PROCESSED_DIR / "pad_components_subcomponents.csv"
ONTOLOGY_PATH = ONTOLOGY_DIR / "innovation_ontology.json"
RESULTS_PATH = OUTPUT_DIR / "innovation_classification_results.csv"
RESULTS_JSONL_PATH = OUTPUT_DIR / "innovation_classification_results.jsonl"

CLASSIFIER_MODEL = "openai/gpt-oss-120b"

# Set False for normal reproducible use.
# Set True only when intentionally rerunning paid/API inference.
RUN_INFERENCE = False

In [ ]:
# --------------------------------------------------
# ENVIRONMENT
# --------------------------------------------------

load_dotenv(ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if RUN_INFERENCE:
    assert GROQ_API_KEY, (
        "GROQ_API_KEY was not found. "
        "Add it to the project-level .env file before running inference."
    )

print("Project root :", ROOT)
print("Run inference:", RUN_INFERENCE)

In [ ]:
# --------------------------------------------------
# LOAD INPUT DATA
# --------------------------------------------------

candidates_df = pd.read_csv(CANDIDATE_PATH)
units_df = pd.read_csv(UNITS_PATH)

with open(ONTOLOGY_PATH, "r", encoding="utf-8") as f:
    INNOVATION_ONTOLOGY = json.load(f)

print("Candidate units       :", len(candidates_df))
print("Full analytical units :", len(units_df))
print("Ontology dimensions   :", list(INNOVATION_ONTOLOGY.keys()))

In [ ]:
# --------------------------------------------------
# PREPARE CLASSIFICATION DATASET
# --------------------------------------------------

def normalize_unit_number(value):
    """Normalize component/subcomponent numbers for reliable joins."""
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.endswith(".0"):
        value = value[:-2]

    return value


candidates_df["number_key"] = (
    candidates_df["number"]
    .apply(normalize_unit_number)
)

units_df["number_key"] = (
    units_df["number"]
    .apply(normalize_unit_number)
)

classification_df = candidates_df.merge(
    units_df[
        [
            "projectid",
            "type",
            "number_key",
            "text",
            "character_count",
        ]
    ],
    on=[
        "projectid",
        "type",
        "number_key",
    ],
    how="left",
    validate="many_to_one",
)

assert len(classification_df) == len(candidates_df)
assert classification_df["text"].notna().all()

print("Classification candidates:", len(classification_df))
print(
    "Candidates with full text:",
    classification_df["text"].notna().sum()
)



classification_df[
    [
        "projectid",
        "type",
        "number",
        "title",
        "retrieval_query",
        "similarity",
    ]
].head()

In [ ]:
# --------------------------------------------------
# SCHEMA AND CLASSIFIER GUIDANCE
# STRUCTURED OUTPUT SCHEMA
# --------------------------------------------------

class InnovationClassification(BaseModel):

    innovation_presence: Literal[
        "innovation",
        "possible_innovation",
        "not_innovation",
        "insufficient_evidence",
    ]

    innovation_types: list[
        Literal[
            "digital_data_technology",
            "process_service_delivery",
            "institutional_governance",
            "financing_instrument",
            "policy_regulatory",
            "product_service",
            "other",
        ]
    ]

    novelty_basis: Literal[
        "explicit_novelty",
        "new_to_context",
        "adaptation",
        "replication",
        "unclear",
    ]

    maturity_stage: Literal[
        "concept",
        "pilot",
        "early_adoption",
        "replication",
        "scaling",
        "mainstreamed",
        "unclear",
    ]

    evidence_strength: Literal[
        "explicit",
        "strongly_inferred",
        "weakly_inferred",
    ]

    evidence_text: str = Field(
        description="Short excerpt or paraphrase identifying the evidence."
    )

    rationale: str = Field(
        description="Concise evidence-based explanation."
    )

    confidence: float = Field(
        ge=0.0,
        le=1.0,
    )



NON_INNOVATION_GUIDANCE = [
    "Routine construction or rehabilitation is not innovation by itself.",
    "Standard training or capacity building is not innovation by itself.",
    "Purchasing computers or equipment is not innovation by itself.",
    "Digitization is not automatically innovation.",
    "Use of common technology is not sufficient evidence of innovation.",
    "Calling an activity innovative does not prove that it is innovative.",
    "Routine project management and monitoring are not innovation by themselves.",
]

In [ ]:
# --------------------------------------------------
# PROMPT DEFINITION
# --------------------------------------------------

SYSTEM_PROMPT = """
You are classifying innovation signals in World Bank project documents.

Be conservative and evidence-based.

An activity is not innovative merely because it:
- uses digital technology,
- provides training,
- purchases equipment,
- builds infrastructure,
- uses monitoring and evaluation,
- or is described as new.

Classify an activity as innovation only when the text shows a materially
different technology, process, institutional arrangement, financing mechanism,
policy approach, product, or service.

Use only the supplied project text.

If novelty cannot be established, prefer:
- possible_innovation,
- not_innovation,
- or insufficient_evidence.

Return valid JSON matching the supplied classification schema.
"""


def build_classification_prompt(row):
    """Build the classifier input for one analytical unit."""

    ontology_text = json.dumps(
        INNOVATION_ONTOLOGY,
        indent=2,
    )

    guidance_text = "\n".join(
        f"- {rule}"
        for rule in NON_INNOVATION_GUIDANCE
    )

    return f"""
INNOVATION ONTOLOGY
{ontology_text}

NON-INNOVATION GUIDANCE
{guidance_text}

PROJECT ID
{row['projectid']}

UNIT
{row['type']} {row['number']}

TITLE
{row['title']}

PROJECT TEXT
{row['text']}

Return a structured JSON classification.
"""

In [ ]:
# --------------------------------------------------
# MODEL CLIENT
# --------------------------------------------------

# initialize client only when needed

client = None

if RUN_INFERENCE:
    client = Groq(
        api_key=GROQ_API_KEY
    )

    print("Classifier model:", CLASSIFIER_MODEL)
else:
    print(
        "Inference disabled. Existing saved results will be loaded."
    )

In [ ]:
# --------------------------------------------------
# inference function
# --------------------------------------------------

def classify_unit(row):
    """Classify one analytical unit."""

    response = client.chat.completions.create(
        model=CLASSIFIER_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": build_classification_prompt(row),
            },
        ],
        response_format={
            "type": "json_object"
        },
        temperature=0,
    )

    raw = response.choices[0].message.content

    return InnovationClassification.model_validate_json(
        raw
    )

In [ ]:
# --------------------------------------------------
# RUN OR LOAD CLASSIFICATIONS
# --------------------------------------------------

if RUN_INFERENCE:

    results = []

    for i, row in classification_df.iterrows():

        print(
            f"[{i + 1}/{len(classification_df)}] "
            f"{row['projectid']} | "
            f"{row['type']} {row['number']}",
            end=" ... "
        )

        try:
            classification = classify_unit(row)

            result = row[
                [
                    "projectid",
                    "type",
                    "number",
                    "title",
                    "similarity",
                ]
            ].to_dict()

            result.update(
                classification.model_dump()
            )

            print("done")

        except Exception as exc:

            result = row[
                [
                    "projectid",
                    "type",
                    "number",
                    "title",
                    "similarity",
                ]
            ].to_dict()

            result.update(
                {
                    "innovation_presence": "error",
                    "innovation_types": [],
                    "novelty_basis": "unclear",
                    "maturity_stage": "unclear",
                    "evidence_strength": "weakly_inferred",
                    "evidence_text": "",
                    "rationale": str(exc),
                    "confidence": 0.0,
                }
            )

            print("error")

        results.append(result)

        time.sleep(1)

    results_df = pd.DataFrame(results)

else:

    assert RESULTS_PATH.exists(), (
        "Saved classification results were not found."
    )

    results_df = pd.read_csv(
        RESULTS_PATH
    )

print("Classification records:", len(results_df))

In [ ]:
# --------------------------------------------------
# SAVE RESULTS
# --------------------------------------------------

if RUN_INFERENCE:

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    results_df.to_csv(
        RESULTS_PATH,
        index=False
    )

    results_df.to_json(
        RESULTS_JSONL_PATH,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print("Saved:", RESULTS_PATH)
    print("Saved:", RESULTS_JSONL_PATH)


# --------------------------------------------------
# CLASSIFICATION SUMMARY
# --------------------------------------------------

summary = (
    results_df["innovation_presence"]
    .value_counts(dropna=False)
    .rename_axis("classification")
    .reset_index(name="count")
)

summary




print(
    "Errors:",
    (results_df["innovation_presence"] == "error").sum()
)

print(
    "\nConfidence summary:"
)

display(
    results_df["confidence"].describe()
)